In [27]:
pip install groq



  Attempting uninstall: typing-extensions

    Found existing installation: typing_extensions 4.12.2

    Uninstalling typing_extensions-4.12.2:

   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------------- 0/2 [typing-extensions]
   ---------------------------------

In [1]:
import os
import sys
import json
import socket
import threading
import base64
from flask import Flask, render_template_string, request, jsonify, session
from IPython.display import display, IFrame
from gtts import gTTS
from io import BytesIO

# Попробуем импортировать Groq
try:
    from groq import Groq
except ImportError:
    print("⚠️ Ошибка: библиотека 'groq' не установлена. Установите её: pip install groq", file=sys.stderr)
    sys.exit(1)

# =====================================================================
# НАСТРОЙКА API GROQ
# =====================================================================
API_KEY = os.environ.get("GROQ_API_KEY", "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")

try:
    client = Groq(api_key=API_KEY)
    print("✅ Groq API настроен")
except Exception as e:
    print(f"❌ Ошибка настройки Groq API: {e}", file=sys.stderr)

# Глобальное хранилище с блокировкой для потокобезопасности
sessions_store = {}
sessions_lock = threading.Lock()

# Используем актуальную модель Groq
DEFAULT_MODEL = "llama-3.1-8b-instant"

SYSTEM_INSTRUCTION = """
Ты — Асад, строгий, слегка циничный, но глубоко профессиональный экзаменатор в техническом вузе.
Твоя цель — проверить реальную глубину понимания студента.
Правила твоего поведения:
1. Студент сдает тебе тему. Слушай его ответ, придирайся к формулировкам и условиям применимости формул.
2. Задавай жесткие дополнительные вопросы: "почему?", "где эта формула ломается?", "какие крайние случаи?".
3. Твои реплики должны быть короткими (1-3 предложения), как в живом устном диалоге.
4. Отвечай строго на русском языке.
"""

app = Flask(__name__)
app.secret_key = 'asad_examiner_secret_key_2024'  # Для Flask session

# =====================================================================
# ENDPOINT ДЛЯ ОЗВУЧКИ
# =====================================================================
@app.route('/api/tts', methods=['POST'])
def text_to_speech():
    data = request.json
    text = data.get('text', '')
    
    if not text:
        return jsonify({'error': 'No text'}), 400
    
    try:
        tts = gTTS(text=text, lang='ru', slow=False)
        audio_buffer = BytesIO()
        tts.write_to_fp(audio_buffer)
        audio_buffer.seek(0)
        audio_base64 = base64.b64encode(audio_buffer.read()).decode('utf-8')
        
        return jsonify({
            'audio': f'data:audio/mpeg;base64,{audio_base64}',
            'text': text
        })
    except Exception as e:
        print(f"❌ Ошибка TTS: {e}")
        return jsonify({'error': str(e)}), 500

HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Асад Экзаменатор</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700;800&family=JetBrains+Mono:wght@700&display=swap" rel="stylesheet">
    <style>
        body { font-family: 'Inter', sans-serif; background-color: #090a0f; }
        .stamp-font { font-family: 'JetBrains Mono', monospace; }
        .glow-red { box-shadow: 0 0 20px rgba(220, 38, 38, 0.4); }
        .pulse-mic { animation: pulse 1.5s infinite; }
        @keyframes pulse {
            0% { transform: scale(1); box-shadow: 0 0 0 0 rgba(220, 38, 38, 0.7); }
            70% { transform: scale(1.05); box-shadow: 0 0 0 15px rgba(220, 38, 38, 0); }
            100% { transform: scale(1); box-shadow: 0 0 0 0 rgba(220, 38, 38, 0); }
        }
        .recording-active {
            background-color: #dc2626 !important;
            animation: pulse 1s infinite;
        }
    </style>
</head>
<body class="text-gray-100 min-h-screen flex flex-col justify-between overflow-x-hidden">

    <div id="screen-start" class="container mx-auto px-4 py-8 max-w-6xl my-auto grid grid-cols-1 lg:grid-cols-12 gap-8">
        <div class="lg:col-span-5 flex flex-col justify-between bg-[#11131e] p-8 rounded-2xl border border-gray-800">
            <div>
                <span class="text-red-500 font-bold uppercase tracking-widest text-xs">ИИ-симулятор экзамена</span>
                <h1 class="text-5xl font-extrabold tracking-tight mt-2 mb-4">АСАД<br><span class="text-red-600">ЭКЗАМЕНАТОР</span></h1>
                <p class="text-gray-400 text-sm mb-6 leading-relaxed">Жесткий спарринг перед сессией.</p>
                <div class="flex flex-wrap gap-2 mb-8">
                    <span class="bg-gray-800 text-gray-300 text-xs px-3 py-1.5 rounded-full">🎤 Голос / ⌨️ Текст</span>
                    <span class="bg-gray-800 text-gray-300 text-xs px-3 py-1.5 rounded-full">Без регистрации</span>
                </div>
            </div>
            <div class="bg-black/40 p-4 rounded-xl border border-gray-800 flex items-center gap-4">
                <div class="w-12 h-12 rounded-full bg-red-950 border border-red-800 flex items-center justify-center text-xl font-bold">А</div>
                <div>
                    <div class="text-sm font-semibold">Готов к допросу?</div>
                    <div class="text-xs text-gray-500">Заходи и покажи уровень.</div>
                </div>
            </div>
        </div>

        <div class="lg:col-span-7 space-y-6">
            <div id="step-1-card" class="bg-[#11131e] p-8 rounded-2xl border border-gray-800 space-y-6">
                <div class="flex justify-between items-center">
                    <span class="text-xs font-semibold text-red-500 uppercase">Шаг 1 из 2</span>
                    <h2 class="text-xl font-bold">Выбери тему</h2>
                </div>
                <div>
                    <label class="block text-xs text-gray-400 mb-2">Введите тему</label>
                    <input type="text" id="topic-input" placeholder="Например: Интегралы" 
                           class="w-full bg-[#090a0f] border border-gray-800 rounded-xl px-4 py-3 text-sm focus:outline-none focus:border-red-600">
                </div>
                <div class="space-y-2">
                    <span class="text-xs text-gray-500">Популярные темы:</span>
                    <div class="grid grid-cols-1 md:grid-cols-2 gap-2">
                        <button onclick="selectPresetTopic('Математический анализ')" class="text-left text-xs bg-[#171a2a] hover:bg-red-950/30 border border-gray-800 rounded-xl px-4 py-3">Математический анализ</button>
                        <button onclick="selectPresetTopic('Линейная Алгебра')" class="text-left text-xs bg-[#171a2a] hover:bg-red-950/30 border border-gray-800 rounded-xl px-4 py-3">Линейная Алгебра</button>
                        <button onclick="selectPresetTopic('Теория вероятностей')" class="text-left text-xs bg-[#171a2a] hover:bg-red-950/30 border border-gray-800 rounded-xl px-4 py-3">Теория вероятностей</button>
                        <button onclick="selectPresetTopic('Алгоритмы')" class="text-left text-xs bg-[#171a2a] hover:bg-red-950/30 border border-gray-800 rounded-xl px-4 py-3">Алгоритмы</button>
                    </div>
                </div>
                <button onclick="goToStep2()" class="w-full bg-red-600 hover:bg-red-700 text-sm font-bold py-3.5 rounded-xl">ДАЛЕЕ</button>
            </div>

            <div id="step-2-card" class="bg-[#11131e] p-8 rounded-2xl border border-gray-800 space-y-6 hidden">
                <div class="flex justify-between items-center">
                    <span class="text-xs font-semibold text-red-500 uppercase">Шаг 2 из 2</span>
                    <h2 class="text-xl font-bold">Проверка оборудования</h2>
                </div>
                <p class="text-xs text-gray-400">Убедитесь, что микрофон работает.</p>
                <div class="flex flex-col items-center py-6 space-y-4">
                    <div id="mic-status-ring" class="w-16 h-16 rounded-full bg-red-950/40 border border-red-800 flex items-center justify-center">
                        <svg class="w-7 h-7 text-red-500" fill="currentColor" viewBox="0 0 20 20"><path d="M7 4a3 3 0 016 0v4a3 3 0 11-6 0V4zm4 10.93A7.001 7.001 0 0017 8a1 1 0 10-2 0A5 5 0 015 8a1 1 0 00-2 0 7.001 7.001 0 006 6.93V17H6a1 1 0 100 2h8a1 1 0 100-2h-3v-2.07z"></path></svg>
                    </div>
                    <span id="mic-status-text" class="text-xs text-gray-400">Проверяем доступ...</span>
                    <button onclick="enableKeyboardMode()" class="text-xs text-red-400 hover:underline">Использовать клавиатуру</button>
                </div>
                <div class="flex gap-3">
                    <button onclick="backToStep1()" class="w-1/3 bg-gray-800 hover:bg-gray-700 text-xs font-bold py-3.5 rounded-xl">НАЗАД</button>
                    <button onclick="startExam()" class="w-2/3 bg-red-600 text-white text-sm font-bold py-3.5 rounded-xl">НАЧАТЬ ДОПРОС</button>
                </div>
            </div>
        </div>
    </div>

    <div id="screen-exam" class="container mx-auto px-4 py-6 max-w-6xl my-auto grid grid-cols-1 lg:grid-cols-12 gap-6 hidden">
        <div class="lg:col-span-8 bg-[#11131e] border border-gray-800 rounded-2xl p-6 space-y-6">
            <div class="flex justify-between items-center border-b border-gray-800 pb-4">
                <div>
                    <span class="text-xs text-red-500 font-semibold uppercase">Тема</span>
                    <h3 id="active-topic-title" class="text-lg font-bold">-</h3>
                </div>
                <div class="flex items-center gap-4">
                    <div id="exam-timer" class="text-sm font-mono bg-gray-900 px-3 py-1.5 rounded-lg border border-gray-800">00:00</div>
                    <button onclick="finishExam()" class="bg-red-600 hover:bg-red-700 text-xs font-bold px-4 py-2 rounded-xl">Завершить</button>
                </div>
            </div>

            <div class="flex gap-4 items-start">
                <div class="w-16 h-16 rounded-2xl bg-gradient-to-br from-red-900 to-black border border-red-700 flex-shrink-0 flex items-center justify-center text-2xl font-black text-red-500">А</div>
                <div class="space-y-1 w-full">
                    <div class="flex items-center gap-2">
                        <span class="font-bold text-sm">Асад Экзаменатор</span>
                        <span id="asad-status" class="w-2 h-2 rounded-full bg-green-500 animate-pulse"></span>
                    </div>
                    <p id="asad-question" class="text-gray-300 text-sm bg-[#090a0f] p-4 rounded-xl border border-gray-800">Загрузка...</p>
                    <audio id="asad-audio" controls class="w-full mt-2 hidden" style="height: 40px;"></audio>
                </div>
            </div>

            <div class="bg-[#090a0f] rounded-xl p-6 border border-gray-800 space-y-4">
                <div id="voice-input-container" class="flex flex-col items-center space-y-4">
                    <button id="record-btn" onclick="toggleSpeechRecord()" class="w-16 h-16 rounded-full bg-red-600 hover:bg-red-700 flex items-center justify-center glow-red">
                        <svg class="w-7 h-7 text-white" fill="none" stroke="currentColor" viewBox="0 0 24 24"><path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M19 11a7 7 0 01-7 7m0 0a7 7 0 01-7-7m7 7v4m0 0H8m4 0h4m-4-8a3 3 0 01-3-3V5a3 3 0 116 0v6a3 3 0 01-3 3z"></path></svg>
                    </button>
                    <div class="text-center">
                        <p id="record-prompt" class="text-xs font-semibold text-gray-300">Нажмите для записи</p>
                        <p id="record-status" class="text-xs text-gray-500 mt-1">Готов</p>
                        <button onclick="switchToTextMode()" class="text-[10px] text-red-400 hover:underline mt-2">Переключить на текст</button>
                    </div>
                    <div id="live-transcript" class="w-full text-xs text-gray-400 bg-black/40 p-3 rounded-lg border border-gray-800 italic text-center hidden"></div>
                    <button id="send-voice-btn" onclick="sendVoiceAnswer()" class="hidden bg-green-600 hover:bg-green-700 text-xs font-bold px-6 py-2.5 rounded-xl">✓ ОТПРАВИТЬ ОТВЕТ</button>
                </div>

                <div id="text-input-container" class="space-y-3 hidden">
                    <textarea id="text-answer" rows="3" placeholder="Введите ответ..." class="w-full bg-[#11131e] border border-gray-800 rounded-xl px-4 py-3 text-sm focus:outline-none focus:border-red-600 text-gray-200"></textarea>
                    <div class="flex justify-between items-center">
                        <button onclick="switchToVoiceMode()" class="text-xs text-gray-500 hover:underline">← К голосу</button>
                        <button onclick="sendTextAnswer()" class="bg-red-600 hover:bg-red-700 text-xs font-bold px-5 py-2.5 rounded-xl">Отправить</button>
                    </div>
                </div>
            </div>
        </div>

        <div class="lg:col-span-4 bg-[#11131e] border border-gray-800 rounded-2xl p-6">
            <h4 class="text-sm font-bold text-gray-300 uppercase mb-6 pb-2 border-b border-gray-800">Статистика</h4>
            <div class="space-y-6">
                <div class="flex justify-between items-center">
                    <div>
                        <span class="text-xs text-gray-400">Паузы</span>
                        <span class="text-xs text-gray-500 block">Задержки</span>
                    </div>
                    <span id="stat-pauses" class="text-xl font-bold text-red-500">0</span>
                </div>
                <div class="flex justify-between items-center">
                    <div>
                        <span class="text-xs text-gray-400">Слова-паразиты</span>
                        <span class="text-xs text-gray-500 block">"ну", "типа"</span>
                    </div>
                    <span id="stat-fillers" class="text-xl font-bold text-red-500">0</span>
                </div>
                <div class="flex justify-between items-center">
                    <div>
                        <span class="text-xs text-gray-400">Уверенность</span>
                        <span class="text-xs text-gray-500 block">Чистота речи</span>
                    </div>
                    <span id="stat-confidence" class="text-xl font-bold text-green-500">100%</span>
                </div>
                <div class="flex justify-between items-center">
                    <div>
                        <span class="text-xs text-gray-400">Ошибки</span>
                        <span class="text-xs text-gray-500 block">Логические</span>
                    </div>
                    <span id="stat-errors" class="text-xl font-bold text-yellow-500">0</span>
                </div>
            </div>
        </div>
    </div>

    <div id="screen-verdict" class="container mx-auto px-4 py-8 max-w-xl my-auto bg-[#11131e] border border-gray-800 rounded-2xl p-8 space-y-8 hidden">
        <div class="text-center space-y-2">
            <span class="text-xs text-red-500 font-bold uppercase tracking-widest">Экзамен завершен</span>
            <h2 class="text-3xl font-extrabold">ВЕРДИКТ</h2>
        </div>
        <div class="flex justify-center my-6">
            <div id="verdict-stamp" class="stamp-font border-4 border-red-600 text-red-600 text-3xl font-black uppercase px-8 py-3 rounded-xl tracking-widest transform -rotate-3 glow-red">ПОКА ЖИВИ</div>
        </div>
        <p id="verdict-desc" class="text-sm text-gray-400 text-center">Обработка...</p>
        <div class="space-y-3">
            <h4 class="text-xs font-bold text-gray-300 uppercase">Логические пробелы:</h4>
            <ul id="verdict-holes" class="space-y-2 text-xs text-gray-400"></ul>
        </div>
        <button onclick="restartApp()" class="w-full bg-red-600 hover:bg-red-700 text-xs font-bold py-4 rounded-xl">ПРОЙТИ ЕЩЕ РАЗ</button>
    </div>

    <script>
        let activeTopic = '';
        let timerInterval = null;
        let examSeconds = 0;
        let recognition = null;
        let isRecording = false;
        let currentTranscript = '';
        let keyboardModeActive = false;
        let pauseCount = 0;
        let fillerCount = 0;
        let errorCount = 0;
        let baseConfidence = 100;
        const fillerWords = ['ну', 'типа', 'как бы', 'короче', 'э-э', 'а-а', 'это самое', 'в общем'];

        if ('webkitSpeechRecognition' in window || 'SpeechRecognition' in window) {
            const SpeechRecognition = window.SpeechRecognition || window.webkitSpeechRecognition;
            recognition = new SpeechRecognition();
            recognition.continuous = false;
            recognition.interimResults = true;
            recognition.lang = 'ru-RU';
            recognition.maxAlternatives = 1;

            recognition.onresult = function(event) {
                let interimTranscript = '';
                let finalTranscript = '';
                for (let i = event.resultIndex; i < event.results.length; ++i) {
                    if (event.results[i].isFinal) finalTranscript += event.results[i][0].transcript;
                    else interimTranscript += event.results[i][0].transcript;
                }
                currentTranscript = finalTranscript || interimTranscript;
                const liveDiv = document.getElementById('live-transcript');
                if (currentTranscript.trim()) {
                    liveDiv.classList.remove('hidden');
                    liveDiv.innerText = '"' + currentTranscript + '"';
                }
                if (finalTranscript) document.getElementById('send-voice-btn').classList.remove('hidden');
                analyzeFillers(currentTranscript);
            };

            recognition.onerror = function(event) {
                console.log("Ошибка:", event.error);
                document.getElementById('record-status').innerText = "Ошибка: " + event.error;
                if (event.error === 'not-allowed') enableKeyboardMode();
                stopRecording();
            };
            
            recognition.onend = function() {
                if (isRecording) {
                    stopRecording();
                    if (currentTranscript.trim()) {
                        document.getElementById('send-voice-btn').classList.remove('hidden');
                        document.getElementById('record-status').innerText = "Нажмите 'Отправить'";
                    }
                }
            };
        }

        function selectPresetTopic(topic) { document.getElementById('topic-input').value = topic; }

        function goToStep2() {
            const topic = document.getElementById('topic-input').value.trim();
            if (!topic) { alert("Укажите тему."); return; }
            activeTopic = topic;
            document.getElementById('step-1-card').classList.add('hidden');
            document.getElementById('step-2-card').classList.remove('hidden');
            navigator.mediaDevices.getUserMedia({ audio: true })
                .then(stream => {
                    document.getElementById('mic-status-ring').classList.add('pulse-mic');
                    document.getElementById('mic-status-text').innerText = "Микрофон подключен";
                    document.getElementById('mic-status-text').className = "text-xs text-green-500";
                    stream.getTracks().forEach(track => track.stop());
                })
                .catch(err => {
                    document.getElementById('mic-status-text').innerText = "Микрофон заблокирован";
                    document.getElementById('mic-status-text').className = "text-xs text-red-500";
                });
        }

        function backToStep1() {
            document.getElementById('step-2-card').classList.add('hidden');
            document.getElementById('step-1-card').classList.remove('hidden');
        }

        function enableKeyboardMode() {
            keyboardModeActive = true;
            document.getElementById('voice-input-container').classList.add('hidden');
            document.getElementById('text-input-container').classList.remove('hidden');
        }

        function switchToTextMode() {
            keyboardModeActive = true;
            document.getElementById('voice-input-container').classList.add('hidden');
            document.getElementById('text-input-container').classList.remove('hidden');
            if (isRecording) stopRecording();
        }

        function switchToVoiceMode() {
            keyboardModeActive = false;
            document.getElementById('text-input-container').classList.add('hidden');
            document.getElementById('voice-input-container').classList.remove('hidden');
        }

        function startExam() {
            document.getElementById('screen-start').classList.add('hidden');
            document.getElementById('screen-exam').classList.remove('hidden');
            document.getElementById('active-topic-title').innerText = activeTopic;
            if (keyboardModeActive) switchToTextMode();
            
            examSeconds = 0;
            timerInterval = setInterval(() => {
                examSeconds++;
                const mins = String(Math.floor(examSeconds / 60)).padStart(2, '0');
                const secs = String(examSeconds % 60).padStart(2, '0');
                document.getElementById('exam-timer').innerText = mins + ':' + secs;
            }, 1000);

            fetch('/api/start', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ topic: activeTopic })
            })
            .then(res => res.json())
            .then(data => {
                console.log('Start response:', data);
                if (data.error) {
                    document.getElementById('asad-question').innerText = "Ошибка: " + data.error;
                    return;
                }
                speakQuestion(data.question);
            })
            .catch(err => {
                console.error('Fetch error:', err);
                document.getElementById('asad-question').innerText = "Ошибка сети: " + err;
            });
        }

        function speakQuestion(text) {
            document.getElementById('asad-question').innerText = text;
            document.getElementById('asad-status').className = "w-2 h-2 rounded-full bg-green-500 animate-pulse";

            fetch('/api/tts', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ text: text })
            })
            .then(res => res.json())
            .then(data => {
                if (data.audio) {
                    const audioElement = document.getElementById('asad-audio');
                    audioElement.src = data.audio;
                    audioElement.classList.remove('hidden');
                    audioElement.play().catch(e => console.log('Autoplay blocked:', e));
                    audioElement.onended = () => {
                        document.getElementById('asad-status').className = "w-2 h-2 rounded-full bg-gray-500";
                    };
                }
            })
            .catch(err => console.log('TTS error:', err));
        }

        function sendTextAnswer() {
            const area = document.getElementById('text-answer');
            const val = area.value.trim();
            if (!val) return;
            area.value = '';
            analyzeFillers(val);
            sendToServer(val);
        }

        function toggleSpeechRecord() {
            if (!recognition) { switchToTextMode(); return; }
            if (!isRecording) startRecording();
            else stopRecording();
        }

        function startRecording() {
            isRecording = true;
            currentTranscript = '';
            const btn = document.getElementById('record-btn');
            btn.classList.add('recording-active');
            document.getElementById('record-prompt').innerText = "Идет запись...";
            document.getElementById('record-status').innerText = "Говорите четко";
            document.getElementById('live-transcript').classList.remove('hidden');
            document.getElementById('live-transcript').innerText = "Слушаю...";
            document.getElementById('send-voice-btn').classList.add('hidden');
            try { recognition.start(); } catch(e) { switchToTextMode(); }
        }

        function stopRecording() {
            isRecording = false;
            document.getElementById('record-btn').classList.remove('recording-active');
            document.getElementById('record-prompt').innerText = "Нажмите для новой записи";
            document.getElementById('record-status').innerText = "Запись завершена";
            try { recognition.stop(); } catch(e) {}
        }

        function sendVoiceAnswer() {
            const textToSend = currentTranscript.trim();
            if (!textToSend) { alert("Не распознано. Попробуйте еще."); return; }
            document.getElementById('send-voice-btn').classList.add('hidden');
            document.getElementById('live-transcript').classList.add('hidden');
            analyzeFillers(textToSend);
            sendToServer(textToSend);
        }

        function sendToServer(transcript) {
            document.getElementById('record-status').innerText = "Отправка...";
            fetch('/api/respond', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ transcript: transcript, pauses: pauseCount, fillers: fillerCount })
            })
            .then(res => res.json())
            .then(data => {
                document.getElementById('record-status').innerText = "Готов";
                if (data.error) {
                    document.getElementById('asad-question').innerText = "Ошибка: " + data.error;
                    return;
                }
                if (data.is_error) {
                    errorCount++;
                    document.getElementById('stat-errors').innerText = errorCount;
                    updateConfidence();
                }
                speakQuestion(data.question);
            })
            .catch(err => {
                console.error('Respond error:', err);
                document.getElementById('record-status').innerText = "Ошибка";
            });
        }

        function analyzeFillers(text) {
            let foundCount = 0;
            text.toLowerCase().split(/\s+/).forEach(word => {
                if (fillerWords.includes(word)) foundCount++;
            });
            if (foundCount > fillerCount) {
                fillerCount = foundCount;
                document.getElementById('stat-fillers').innerText = fillerCount;
                updateConfidence();
            }
        }

        function updateConfidence() {
            let conf = baseConfidence - (fillerCount * 4) - (pauseCount * 6) - (errorCount * 15);
            if (conf < 20) conf = 20;
            const confSpan = document.getElementById('stat-confidence');
            confSpan.innerText = conf + '%';
            if (conf < 50) confSpan.className = "text-xl font-bold text-red-500";
            else if (conf < 80) confSpan.className = "text-xl font-bold text-yellow-500";
            else confSpan.className = "text-xl font-bold text-green-500";
        }

        function finishExam() {
            if (timerInterval) clearInterval(timerInterval);
            if (isRecording) stopRecording();
            document.getElementById('screen-exam').classList.add('hidden');
            document.getElementById('screen-verdict').classList.remove('hidden');
            document.getElementById('verdict-desc').innerText = "Асад оценивает...";

            fetch('/api/finish', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ pauses: pauseCount, fillers: fillerCount, errors: errorCount })
            })
            .then(res => res.json())
            .then(data => {
                const stamp = document.getElementById('verdict-stamp');
                stamp.innerText = data.verdict;
                if (data.verdict === 'НЕ СДАНО') stamp.className = "stamp-font border-4 border-red-600 text-red-600 text-3xl font-black uppercase px-8 py-3 rounded-xl tracking-widest transform -rotate-3 glow-red";
                else if (data.verdict === 'ПОКА ЖИВИ') stamp.className = "stamp-font border-4 border-yellow-500 text-yellow-500 text-3xl font-black uppercase px-8 py-3 rounded-xl tracking-widest transform -rotate-3";
                else stamp.className = "stamp-font border-4 border-green-500 text-green-500 text-3xl font-black uppercase px-8 py-3 rounded-xl tracking-widest transform -rotate-3";
                document.getElementById('verdict-desc').innerText = data.description;
                const holesUl = document.getElementById('verdict-holes');
                holesUl.innerHTML = '';
                data.holes.forEach((hole, index) => {
                    const li = document.createElement('li');
                    li.className = "flex items-start gap-2 text-xs text-gray-400";
                    li.innerHTML = `<span class="text-red-500 font-bold">${index + 1}.</span> <span>${hole}</span>`;
                    holesUl.appendChild(li);
                });
            });
        }

        function restartApp() {
            document.getElementById('screen-verdict').classList.add('hidden');
            document.getElementById('screen-start').classList.remove('hidden');
            document.getElementById('step-2-card').classList.add('hidden');
            document.getElementById('step-1-card').classList.remove('hidden');
            pauseCount = 0; fillerCount = 0; errorCount = 0;
            document.getElementById('stat-pauses').innerText = 0;
            document.getElementById('stat-fillers').innerText = 0;
            document.getElementById('stat-errors').innerText = 0;
            document.getElementById('stat-confidence').innerText = '100%';
            document.getElementById('stat-confidence').className = "text-xl font-bold text-green-500";
        }
    </script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

# =====================================================================
# API ЭНДПОИНТЫ
# =====================================================================

@app.route('/api/start', methods=['POST'])
def start_session():
    try:
        data = request.json
        topic = data.get('topic', 'Математика')
        
        import uuid
        session_id = str(uuid.uuid4())
        
        messages = [
            {"role": "system", "content": SYSTEM_INSTRUCTION},
            {"role": "user", "content": f"Привет. Я студент, сдаю тебе тему: {topic}. Задай мне свой первый вопрос по ней."}
        ]
        
        # Запрос к Groq API с использованием актуальной модели
        response = client.chat.completions.create(
            model=DEFAULT_MODEL,
            messages=messages,
            temperature=0.7
        )
        first_question = response.choices[0].message.content
        
        messages.append({"role": "assistant", "content": first_question})
        
        with sessions_lock:
            sessions_store[session_id] = {
                'history': messages,
                'topic': topic
            }
        
        print(f"✅ Сессия создана: {session_id}, тема: {topic}")
        
        return jsonify({
            'session_id': session_id,
            'question': first_question
        })
    except Exception as e:
        print(f"❌ Ошибка /api/start: {e}")
        return jsonify({'error': str(e)}), 500

@app.route('/api/respond', methods=['POST'])
def respond():
    try:
        data = request.json
        transcript = data.get('transcript', '')
        
        with sessions_lock:
            if not sessions_store:
                return jsonify({'error': 'Сессия не найдена. Начните экзамен заново.'}), 404
            
            session_id = list(sessions_store.keys())[-1]
            session_data = sessions_store[session_id]
            history = session_data['history']
            topic = session_data['topic']
        
        history.append({"role": "user", "content": f"Студент отвечает: {transcript}"})
        
        response = client.chat.completions.create(
            model=DEFAULT_MODEL,
            messages=history,
            temperature=0.7
        )
        next_question = response.choices[0].message.content
        
        history.append({"role": "assistant", "content": next_question})
        
        error_evaluator_prompt = f"""
        Тема экзамена: {topic}
        Ответ студента: {transcript}
        
        Правдив и точен ли этот ответ? Дай вердикт строго в формате JSON:
        {{"is_incorrect": true/false}}
        Не используй разметку markdown, верни только чистый JSON.
        """
        
        is_incorrect = False
        try:
            eval_response = client.chat.completions.create(
                model=DEFAULT_MODEL,
                messages=[{"role": "user", "content": error_evaluator_prompt}],
                response_format={"type": "json_object"},
                temperature=0.1
            )
            eval_data = json.loads(eval_response.choices[0].message.content.strip())
            is_incorrect = eval_data.get('is_incorrect', False)
        except Exception as e:
            print(f"⚠️ Ошибка оценки правильности: {e}")
            
        return jsonify({
            'question': next_question,
            'is_error': is_incorrect
        })
    except Exception as e:
        print(f"❌ Ошибка /api/respond: {e}")
        return jsonify({'error': str(e)}), 500

@app.route('/api/finish', methods=['POST'])
def finish_session():
    try:
        data = request.json
        pauses = data.get('pauses', 0)
        fillers = data.get('fillers', 0)
        errors = data.get('errors', 0)
        
        with sessions_lock:
            if not sessions_store:
                return jsonify({
                    'verdict': 'НЕ СДАНО',
                    'description': 'Сессия не найдена.',
                    'holes': ['Начните экзамен заново']
                })
            
            session_id = list(sessions_store.keys())[-1]
            session_data = sessions_store[session_id]
            history = session_data['history']
            topic = session_data['topic']
        
        verdict_prompt = f"""
        Экзамен на тему '{topic}' завершен.
        Статистика: пауз: {pauses}, слов-паразитов: {fillers}, ошибок: {errors}
        
        На основе этой статистики и прошедшего диалога выяви 2-4 логических пробела. 
        Присвой вердикт: 'НЕ СДАНО' или 'ПОКА ЖИВИ' или 'КРАСАВЧИК'
        
        Дай вердикт строго в формате JSON:
        {{"verdict": "...", "description": "...", "holes": ["...", "..."]}}
        Не используй разметку markdown (без ```json), верни только чистый JSON.
        """
        
        try:
            verdict_response = client.chat.completions.create(
                model=DEFAULT_MODEL,
                messages=history + [{"role": "user", "content": verdict_prompt}],
                response_format={"type": "json_object"},
                temperature=0.5
            )
            verdict_data = json.loads(verdict_response.choices[0].message.content.strip())
        except Exception as e:
            print(f"⚠️ Ошибка формирования вердикта: {e}")
            verdict_data = {
                'verdict': 'ПОКА ЖИВИ' if errors < 3 else 'НЕ СДАНО',
                'description': 'Экзамен завершен с техническими трудностями при парсинге результата.',
                'holes': ['Неуверенные речевые паттерны.', 'Не раскрыты условия теорем.']
            }
            
        with sessions_lock:
            if session_id in sessions_store:
                del sessions_store[session_id]
        
        return jsonify(verdict_data)
    except Exception as e:
        print(f"❌ Ошибка /api/finish: {e}")
        return jsonify({'error': str(e)}), 500

def get_free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = get_free_port()

def run_flask():
    import logging
    log = logging.getLogger('werkzeug')
    log.setLevel(logging.ERROR)
    app.run(port=port, debug=False, use_reloader=False, threaded=True)

threading.Thread(target=run_flask, daemon=True).start()

print(f"✅ Приложение запущено на порту {port}!")
print(f"🔗 Откройте: http://127.0.0.1:{port}")
print(f"📝 Логи будут выводиться в консоль Jupyter")

display(IFrame(src=f"http://127.0.0.1:{port}", width="100%", height="750px"))

✅ Groq API настроен
✅ Приложение запущено на порту 4073!
🔗 Откройте: http://127.0.0.1:4073
📝 Логи будут выводиться в консоль Jupyter


 * Serving Flask app '__main__'
 * Debug mode: off
✅ Сессия создана: 0193b1df-63b8-44d4-944d-1a5531528aa0, тема: Математический анализ
✅ Сессия создана: 4d0d4254-a0d5-459f-9157-0cba4446f0c9, тема: Математический анализ
